# GroupBy & Aggregation
** Data Analysis**

---

## What is GroupBy?

GroupBy lets you **split** a DataFrame into groups, **apply** a function to each group, and **combine** the results back. This is called the **Split-Apply-Combine** pattern.

Real-world example: you have sales data for multiple regions. You want the total revenue per region. GroupBy splits the data by region, applies `.sum()` to each group, then combines the results into one table.

Without GroupBy you'd write loops. With GroupBy it's one line.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name":       ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace"],
    "department": ["Eng", "Marketing", "Eng", "HR", "Eng", "Marketing", "HR"],
    "salary":     [90000, 72000, 105000, 68000, 88000, 76000, 71000],
    "years_exp":  [2, 5, 10, 3, 1, 4, 6],
    "rating":     [4.5, 3.8, 4.9, 4.1, 3.5, 4.2, 4.7],
})
df

,name,department,salary,years_exp,rating
0,Alice,Eng,90000,2,4.5
1,Bob,Marketing,72000,5,3.8
2,Charlie,Eng,105000,10,4.9
3,Diana,HR,68000,3,4.1
4,Eve,Eng,88000,1,3.5
5,Frank,Marketing,76000,4,4.2
6,Grace,HR,71000,6,4.7


---
## 1. Basic GroupBy — Brief

`df.groupby("col")` creates a GroupBy object — think of it as the data split into buckets, one per unique value in that column. Nothing is computed yet. You get the result when you call an aggregation method on it.

# Pandas `groupby` — Split → Apply → Combine

## What `groupby()` Actually Returns

When you call `df.groupby("col")`, pandas **does not compute anything yet**. It creates a `GroupBy` object — essentially a map of which rows belong to which group.

```python
grouped = df.groupby("department")
print(type(grouped))   # <pandas.core.groupby.DataFrameGroupBy object>
```

You can inspect this map directly:

```python
grouped.groups
# {
#   'Eng':       [0, 2, 4],   ← row indices of Eng employees
#   'HR':        [3, 6],
#   'Marketing': [1, 5]
# }
```

## The Full Flow

```
df.groupby("department")   →   ["salary"]   →   .mean()
        ↑                          ↑                ↑
  split into buckets         pick the column    apply function
  (no computation yet)       to work on         to each bucket
                                                 then combine
```

## Concrete Example

```python
df = pd.DataFrame({
    "name":       ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "department": ["Eng",   "Mkt", "Eng",     "HR",    "Eng"],
    "salary":     [90000,   72000,  105000,   68000,   88000],
})
```

Step by step what happens inside `df.groupby("department")["salary"].mean()`:

```
Split:
  Eng bucket    → [90000, 105000, 88000]   (Alice, Charlie, Eve)
  HR bucket     → [68000]                  (Diana)
  Mkt bucket    → [72000]                  (Bob)

Apply .mean() to each bucket:
  Eng    → (90000 + 105000 + 88000) / 3 = 94333
  HR     → 68000
  Mkt    → 72000

Combine into one Series:
  department
  Eng    94333.0
  HR     68000.0
  Mkt    72000.0
```

## What Happens to the `department` Column?

`department` is a regular column in the original DataFrame. After `groupby()`, pandas **promotes it to the index** of the result — it is no longer a regular column.

```
Original df:                     After .groupby("department")["salary"].mean():

name     department  salary       department       ← now the index, not a column
Alice    Eng         90000        Eng      94333
Bob      Mkt         72000        HR       68000
Charlie  Eng         105000       Mkt      72000
Diana    HR          68000
Eve      Eng         88000
```

Use `reset_index()` to bring `department` back as a regular column:

```python
df.groupby("department")["salary"].mean().reset_index()

#   department   salary
# 0    Eng       94333
# 1    HR        68000
# 2    Mkt       72000
```

Now it's a normal DataFrame again — ready to filter, merge, or export.

In [4]:
# Average salary per department
print(df.groupby("department")["salary"].mean())

department
Eng          94333.333333
HR           69500.000000
Marketing    74000.000000
Name: salary, dtype: float64


In [5]:
# Count of employees per department
print(df.groupby("department")["name"].count())

department
Eng          3
HR           2
Marketing    2
Name: name, dtype: int64


In [7]:
# Other common aggregations — min, max, sum, std
print(df.groupby("department")["salary"].max())
print("*" * 20)
print(df.groupby("department")["salary"].min())
print("*" * 20)
print(df.groupby("department")["salary"].std())

department
Eng          105000
HR            71000
Marketing     76000
Name: salary, dtype: int64
********************
department
Eng          88000
HR           68000
Marketing    72000
Name: salary, dtype: int64
********************
department
Eng          9291.573243
HR           2121.320344
Marketing    2828.427125
Name: salary, dtype: float64


---
## 2. ⚠️ `.agg()` — Multiple Aggregations at Once

When you need more than one aggregation per group, use `.agg()` instead of chaining multiple calls.

`.agg()` accepts:
- A list of function names: `["mean", "min", "max"]` — applies all to the selected column
- A dict: `{"col1": "mean", "col2": "sum"}` — different aggregation per column
- Named aggregations (clearest syntax): `new_name=("col", "func")` — you control the output column names

The **named aggregation** syntax is the most readable and the one you should default to:

In [8]:
# List of functions — applies all to salary
print(df.groupby("department")["salary"].agg(["mean", "min", "max", "count"]))

                    mean    min     max  count
department                                    
Eng         94333.333333  88000  105000      3
HR          69500.000000  68000   71000      2
Marketing   74000.000000  72000   76000      2


In [9]:
# Dict — different aggregation per column
print(df.groupby("department").agg({
    "salary":   "mean",
    "years_exp": "sum",
    "rating":   "max",
}))


                  salary  years_exp  rating
department                                 
Eng         94333.333333         13     4.9
HR          69500.000000          9     4.7
Marketing   74000.000000          9     4.2


In [12]:
# Named aggregations — cleanest, you control the output column names
# Syntax: new_col_name=("source_col", "function")
result = df.groupby("department").agg(
    avg_salary   = ("salary",   "mean"),
    max_salary   = ("salary",   "max"),
    total_exp    = ("years_exp", "sum"),
    avg_rating   = ("rating",   "mean"),
    headcount    = ("name",     "count"),
)
result

,avg_salary,max_salary,total_exp,avg_rating,headcount
department,,,,,
Eng,94333.333333,105000,13,4.3,3
HR,69500.000000,71000,9,4.4,2
Marketing,74000.000000,76000,9,4.0,2


---
## 3. ⚠️ `reset_index()` — Turning the Group Key Back into a Column

After a `groupby().agg()`, the group column (e.g. `department`) becomes the **index** of the result — not a regular column. This trips people up when they try to filter or access it.

`.reset_index()` moves the index back into a regular column and resets the index to 0, 1, 2...

```
After groupby:               After reset_index():
             avg_salary        department  avg_salary
department                  0  Eng          94333
Eng           94333         1  HR           69500
HR            69500         2  Marketing    74000
Marketing     74000
```
Use `reset_index()` whenever you want to treat the result like a normal DataFrame — filter it, merge it, save it.

In [13]:
summary = df.groupby("department").agg(
    avg_salary = ("salary", "mean"),
    headcount  = ("name",   "count"),
).reset_index()   # department becomes a regular column

print(summary)
print(summary.dtypes)

# Now you can filter it like a normal DataFrame
summary[summary["avg_salary"] > 80000]

  department    avg_salary  headcount
0        Eng  94333.333333          3
1         HR  69500.000000          2
2  Marketing  74000.000000          2
department        str
avg_salary    float64
headcount       int64
dtype: object


,department,avg_salary,headcount
0,Eng,94333.333333,3


---
## 4. ⚠️ `transform()` vs `agg()` — The Key Difference

This is the most confusing part of GroupBy. Both work on groups, but they return different shapes:

| | `agg()` | `transform()` |
|---|---|---|
| Output shape | **One row per group** (reduced) | **Same shape as input** (not reduced) |
| Use when | You want a summary table | You want to add a column back to the original df |

**`agg()` example:** 7 employees, 3 departments → result has 3 rows (one per department)

**`transform()` example:** 7 employees, 3 departments → result still has 7 rows (each employee gets their department's value)

The classic use case for `transform()`: add a column showing each employee's salary relative to their department's average. You need the department average repeated for every employee in that department — that's only possible with `transform()`.

## `agg` vs `transform`

The key is understanding what `agg` vs `transform` returns.

```python
df.groupby("department")["salary"].agg("mean")
# Returns 3 rows (one per department) — collapsed
# department
# Eng          94333
# HR           69500
# Marketing    74000
```

```python
df.groupby("department")["salary"].transform("mean")
# Returns same number of rows as original df — NOT collapsed
# 0    94333   ← Eng employee
# 1    94333   ← Eng employee
# 2    94333   ← Eng employee
# 3    69500   ← HR employee
# 4    69500   ← HR employee
# ...
```

`transform` broadcasts the group result back to every row that belongs to that group.
Why is this useful? You can directly assign it as a new column:

```python
df["dept_avg_salary"] = df.groupby("department")["salary"].transform("mean")
```

Now every row knows its department's average — and you can do things like:

```python
df["above_avg"] = df["salary"] > df["dept_avg_salary"]
```

With `agg` you'd have to do a separate merge/join to get that column back. `transform` skips that entirely.

In [14]:
# agg() — reduces to one row per group
dept_avg = df.groupby("department")["salary"].agg("mean")
print("agg() result shape:", dept_avg.shape)   # (3,) — 3 departments
print(dept_avg)

agg() result shape: (3,)
department
Eng          94333.333333
HR           69500.000000
Marketing    74000.000000
Name: salary, dtype: float64


In [ ]:
# transform() — keeps same shape as original df
dept_avg_per_employee = df.groupby("department")["salary"].transform("mean")
print("transform() result shape:", dept_avg_per_employee.shape)   # (7,) — all employees
print(dept_avg_per_employee)

transform() result shape: (7,)
0    94333.333333
1    74000.000000
2    94333.333333
3    69500.000000
4    94333.333333
5    74000.000000
6    69500.000000
Name: salary, dtype: float64


In [16]:
# Practical use: add department average and compute how each employee compares
df["dept_avg_salary"]   = df.groupby("department")["salary"].transform("mean")
df["salary_vs_dept_avg"] = df["salary"] - df["dept_avg_salary"]

df[["name", "department", "salary", "dept_avg_salary", "salary_vs_dept_avg"]]

,name,department,salary,dept_avg_salary,salary_vs_dept_avg
0,Alice,Eng,90000,94333.333333,-4333.333333
1,Bob,Marketing,72000,74000.000000,-2000.000000
2,Charlie,Eng,105000,94333.333333,10666.666667
3,Diana,HR,68000,69500.000000,-1500.000000
4,Eve,Eng,88000,94333.333333,-6333.333333
5,Frank,Marketing,76000,74000.000000,2000.000000
6,Grace,HR,71000,69500.000000,1500.000000


---
## 5. Multi-Level GroupBy — Brief

Pass a list of columns to group by multiple levels. The result has a MultiIndex (multiple levels in the index).
Use `reset_index()` to flatten it back to a normal DataFrame.

Pass a list to `groupby()` — the order of the list determines the index hierarchy.
First column = outer (primary) group, second = inner (secondary) group.

Rule: the question you're asking determines the order.

"breakdown per department?" → ["department", "level"]  
"breakdown per level?" → ["level", "department"]

In [18]:
# Add a second grouping column
df["level"] = ["Junior", "Senior", "Senior", "Junior", "Junior", "Senior", "Senior"]

# group by two columns without reset index
print(df.groupby(["department", "level"])["salary"].mean())

# Group by two columns
df.groupby(["department", "level"])["salary"].mean().reset_index()

department  level 
Eng         Junior     89000.0
            Senior    105000.0
HR          Junior     68000.0
            Senior     71000.0
Marketing   Senior     74000.0
Name: salary, dtype: float64


,department,level,salary
0,Eng,Junior,89000.0
1,Eng,Senior,105000.0
2,HR,Junior,68000.0
3,HR,Senior,71000.0
4,Marketing,Senior,74000.0


---
## 6. Custom Aggregation with a Function — Brief

When built-in functions aren't enough, pass a custom function to `.agg()`.

```python
x is the group's column values — a pandas Series containing only the rows for that group.

When pandas runs your lambda, it calls it once per group, passing that group's column data as x.


# For "salary" column, grouped by department:

# When group = Eng,       x = Series([80000, 105000, 100000])
# When group = HR,        x = Series([65000, 68000])
# When group = Marketing, x = Series([78000, 76000])
So in your lambdas:


lambda x: x.max() - x.min()
# Eng:       105000 - 80000  = 25000
# HR:        68000  - 65000  = 3000

How to know what x is each time — the rule:

Whatever column you name in the tuple, x is a Series of that column's values for one group.


df.groupby("department").agg(
    result = ("salary", lambda x: ...)   # x = salary values for one dept
    result = ("age",    lambda x: ...)   # x = age values for one dept
    result = ("rating", lambda x: ...)   # x = rating values for one dept
)
x always = (column you specified) × (rows in current group). Nothing more.
```

In [19]:
# Salary range (max - min) per department
df.groupby("department")["salary"].agg(lambda x: x.max() - x.min())

department
Eng          17000
HR            3000
Marketing     4000
Name: salary, dtype: int64

In [20]:
# Named version — cleaner
df.groupby("department").agg(
    salary_range = ("salary", lambda x: x.max() - x.min()),
    above_avg    = ("salary", lambda x: (x > x.mean()).sum()),  # count above average
).reset_index()

,department,salary_range,above_avg
0,Eng,17000,1
1,HR,3000,1
2,Marketing,4000,1


---
## 7. Practice

In [24]:
sales = pd.DataFrame({
    "product":  ["Widget", "Gadget", "Widget", "Doohickey", "Gadget", "Widget", "Doohickey"],
    "region":   ["North", "South", "South", "North", "North", "East", "East"],
    "units":    [150, 200, 175, 90, 210, 130, 80],
    "price":    [29.99, 49.99, 29.99, 19.99, 49.99, 29.99, 19.99],
    "month":    ["Jan", "Jan", "Feb", "Jan", "Feb", "Feb", "Mar"],
})
sales["revenue"] = sales["units"] * sales["price"]
sales

,product,region,units,price,month,revenue
0,Widget,North,150,29.99,Jan,4498.50
1,Gadget,South,200,49.99,Jan,9998.00
2,Widget,South,175,29.99,Feb,5248.25
3,Doohickey,North,90,19.99,Jan,1799.10
4,Gadget,North,210,49.99,Feb,10497.90
5,Widget,East,130,29.99,Feb,3898.70
6,Doohickey,East,80,19.99,Mar,1599.20


In [26]:
# 1. Total revenue per product 
sales.groupby("product")["revenue"].sum()


product
Doohickey     3398.30
Gadget       20495.90
Widget       13645.45
Name: revenue, dtype: float64

In [ ]:
# 2. For each region: total units, average price, max revenue — use named aggregations
sales.groupby("region").agg(
    total_units = ("units", "sum"),
    average_price = ("price", "mean"),
    max_revenue = ("revenue", "max")
)

,total_units,average_price,max_revenue
region,,,
East,210,24.990000,3898.7
North,450,33.323333,10497.9
South,375,39.990000,9998.0


In [29]:
# 3. Add a column 'avg_revenue_by_product' showing each product's average revenue
#    Hint: use transform
sales["avg_revenue_by_product"] = sales.groupby("product")["revenue"].transform("mean")
print(sales)

     product region  units  price month   revenue  avg_revenue_by_product
0     Widget  North    150  29.99   Jan   4498.50             4548.483333
1     Gadget  South    200  49.99   Jan   9998.00            10247.950000
2     Widget  South    175  29.99   Feb   5248.25             4548.483333
3  Doohickey  North     90  19.99   Jan   1799.10             1699.150000
4     Gadget  North    210  49.99   Feb  10497.90            10247.950000
5     Widget   East    130  29.99   Feb   3898.70             4548.483333
6  Doohickey   East     80  19.99   Mar   1599.20             1699.150000


In [ ]:
# 4. Group by product AND region, get total revenue. Reset index at the end.
sales.groupby(["product", "region"])["revenue"].sum().reset_index()

,product,region,revenue
0,Doohickey,East,1599.20
1,Doohickey,North,1799.10
2,Gadget,North,10497.90
3,Gadget,South,9998.00
4,Widget,East,3898.70
5,Widget,North,4498.50
6,Widget,South,5248.25


In [ ]:
# 5. Which product has the highest revenue range (max - min)? Use a custom lambda.
sales.groupby("product").agg({
    "revenue": lambda x:x.max()-x.min()
}).idxmax()

revenue    Widget
dtype: str

---
## Quick Reference

```python
# Basic
df.groupby("col")["target"].mean()        # one aggregation
df.groupby("col")["target"].agg(["mean", "max", "count"])  # multiple on one column

# Named aggregations — cleanest, use this by default
df.groupby("col").agg(
    new_name = ("source_col", "func"),
    new_name2 = ("source_col2", "func2"),
).reset_index()

# agg vs transform
df.groupby("col")["target"].agg("mean")        # → one row per group
df.groupby("col")["target"].transform("mean")  # → same shape as df, use to add back as column

# Multi-level groupby
df.groupby(["col1", "col2"])["target"].mean().reset_index()

# Custom function
df.groupby("col")["target"].agg(lambda x: x.max() - x.min())

# reset_index — always use after groupby to get a normal DataFrame back
.reset_index()

| `.idxmax()` | index label of max value |
| `.idxmin()` | index label of min value |
```

---
